# Recommend V4 - Qwen3-Embedding-0.6B, rebalanced re-ranking

Fourth notebook in the series. Where the others compared and evaluated,
this one is meant to be **the next production candidate** - the decisions
below consolidate everything learned from the model comparison, the
leave-one-out evaluation (and its popularity-contamination finding), and
the LLM-judge sanity check on real Qwen3 recommendations.

## Decisions this notebook implements

1. **Model: `Qwen3-Embedding-0.6B`.** Won or tied on every trustworthy
   measure from the model comparison (coverage, hub concentration on two
   of three known hubs, LOO-MRR though that metric is caveated) except a
   small diversity edge for `bge-m3` (0.83 vs 0.80) and compute time.
2. **Language condition: `translated_en` only.** The 2x3 grid showed no
   detectable aggregate difference from `native` for any model - not worth
   the ~2x embedding cost going forward.
3. **`FIELD_BOOST` raised from 0.03 to 0.15.** The LLM-judge sanity check
   on the real Qwen3 run found the actual mechanism behind the
   "Creating Your IT Strategy" hub (121/1059 employees, including several
   in unrelated departments): `FIELD_BOOST`'s old ceiling (0.03) was
   structurally outgunned by the *combined* ceiling of the generic,
   non-topical signals - `LEVEL_MATCH_BOOST` + `LANGUAGE_MATCH_BOOST` +
   `TRAINING_TYPE_BOOST` = 0.05 + 0.07 + 0.05 = 0.17. A field match could
   be doing its job correctly and still lose to level+language+format
   alone. 0.15 puts field-match's ceiling in the same range as that
   combined total, so genuine topical relevance can compete with, rather
   than be dominated by, generic matching.
4. **The old domain gate (`-0.08` flat penalty on exact-zero field+preference
   match) is replaced with a continuous scaling mechanism** - see section 7.
   This targets the same root cause the gate was aimed at, but fixes its
   known problem (an exact-zero threshold that a spurious tiny field score
   could dodge - `PROJECT_JOURNAL.md` Phase 19) by removing the threshold
   entirely rather than tightening it.
5. **Training-type boost kept, unchanged mechanism, explicitly documented
   as a confirmed no-op** - ablation showed identical LOO metrics with it
   on or off for all 3 models, because it currently applies the same flat
   amount to every course for a given employee (no `Training_Type` column
   exists in the real catalog to vary against, since it's all LinkedIn
   Learning / effectively online-only). Kept so it activates automatically,
   with zero further code changes, if a future catalog import adds real
   training-format variety.
6. **Logistic regression comparison: dropped, not part of this notebook.**
   It would have trained on the same real-completion labels that leave-one-out
   showed are dominated by a mass-assigned training campaign - training a
   model on that data risks learning "predict the popular assigned course"
   rather than anything about personalization, the same failure mode as
   the leave-one-out ground truth itself.

## What this notebook deliberately does NOT fix

- **Course-purpose mismatches** (e.g. "Careers in Commercial Banking" - a
  career-*entry* course - recommended to employees already established in
  Commercial & Growth / Finance & Performance roles). The LLM-judge check
  found this isn't a weak-signal problem (field-match was genuinely
  contributing, not near-zero) - it's the catalog not distinguishing
  "entry-level career guidance" from "skill-building for existing
  practitioners" as a course attribute. No signal in the data can fix
  this; it needs better catalog metadata. Documented as a named,
  deliberate limitation, not silently left as an unexplained gap.
- **Weight magnitudes are still not empirically tuned** - `FIELD_BOOST`'s
  new value (0.15) is a reasoned adjustment targeting a diagnosed
  mechanism, not a result of optimization against a validated metric.
  Given the leave-one-out ground truth's popularity contamination, there
  currently isn't a trustworthy quantitative target to tune against - this
  remains an open, named limitation rather than a solved problem.

## Verification built into this notebook, not left for later

Per the project's standing practice of checking a fix actually changed
what it was meant to change (rather than assuming it worked): section 10
reruns the same three sanity-check cases and the sparse-profile blind-spot
measurement from the LLM-judge review, and section 11 compares this run's
recommendations directly against the **existing**
`recommendations_translated_en_qwen3-embedding-0.6b.csv` (V4's
predecessor's output) to show concretely what changed.

## 1. Setup & config

In [ ]:
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)

BASE = Path.cwd()
if not (BASE / "data" / "processed").is_dir():
    BASE = BASE.parent

import sys
# recommender_core.py / recommender_config.yaml live alongside this notebook
# (notebooks/test) -- adjust this path if you move either file.
CORE_DIR = BASE / "notebooks" / "test"
sys.path.insert(0, str(CORE_DIR))
import recommender_core as core

config = core.load_config(CORE_DIR / "recommender_config.yaml")
to_plain_array = core.to_plain_array  # kept as a local alias, used directly in several cells below

DATA_PROCESSED = BASE / "data" / "processed"
COMPARISON_OUTPUTS = BASE / "outputs" / "model_comparison"   # V4's predecessor's output, for the before/after diff
OUTPUTS = BASE / "outputs" / "v4"
OUTPUTS.mkdir(parents=True, exist_ok=True)

EMPLOYEES_PATH = DATA_PROCESSED / "employees_final.csv"
CATALOG_FULL_PATH = DATA_PROCESSED / "catalog_full_en_fr_translated.csv"
HISTORY_PATH = DATA_PROCESSED / "history_cleaned.csv"
PREVIOUS_RECOMMENDATIONS_PATH = COMPARISON_OUTPUTS / "recommendations_translated_en_qwen3-embedding-0.6b.csv"

TOP_K = config["ranking"]["top_k"]
MODEL_NAME = config["model"]["name"]


## 2. Load the three real files

In [ ]:
employees = pd.read_csv(EMPLOYEES_PATH).fillna("")
catalog_full = pd.read_csv(CATALOG_FULL_PATH).fillna("")
history = pd.read_csv(HISTORY_PATH).fillna("")

assert employees["Email"].is_unique, "Employee emails should be unique"
print("employees:", employees.shape)
print("catalog_full:", catalog_full.shape)
print("history:", history.shape)


## 3. Derive the active-only candidate pool (`translated_en` only)

In [ ]:
n_before = len(catalog_full)
catalog_pool = catalog_full[
    (catalog_full["Status"].str.lower() == "active")
    & (catalog_full["Level"].str.lower() != "general")
].copy().reset_index(drop=True)

catalog_pool["_level_norm"] = catalog_pool["Level"].str.lower().str.strip()
catalog_pool["_lang"] = catalog_pool["Library"].str.upper().map({"EN-US": "EN", "FR-FR": "FR"}).fillna("")
catalog_pool["_text"] = (
    catalog_pool["Course Title_EN"].astype(str) + ". "
    + catalog_pool["Subjects_EN"].astype(str) + ". "
    + catalog_pool["Skills_EN"].astype(str) + ". "
    + catalog_pool["Description_EN"].astype(str)
).str.strip()

course_id_index = {str(cid): idx for idx, cid in enumerate(catalog_pool["Course ID"])}
n_course = len(catalog_pool)
print(f"Full catalog: {n_before} -> candidate pool: {n_course} rows")


## 4. Per-employee history signals (`Excluded_Course_IDs`, `Inferred_Language`)

Identical mechanism to V3 / the comparison notebook. Unlike
`Recommend_Evaluation.ipynb`, `Skills` is used as-is from
`employees_final.csv` -- this notebook scores the real population directly,
it isn't leave-one-out, so there's no held-out completion to hide.

In [ ]:
# LANGUAGE_THRESHOLD now lives in recommender_config.yaml as language.dominance_threshold
LANGUAGE_THRESHOLD = config["language"]["dominance_threshold"]
id_to_library = dict(zip(catalog_full["Course ID"].astype(str), catalog_full["Library"]))

history["_course_id"] = history["ID de contenu"].astype(str)
history["_status_norm"] = history["Status"].str.strip().str.lower()

def derive_employee_signals(group):
    completed = group[group["_status_norm"] == "completed"]
    in_progress = group[group["_status_norm"] == "in-progress"]
    excluded_ids = set(completed["_course_id"]).union(set(in_progress["_course_id"]))

    langs = completed["_course_id"].map(id_to_library).dropna()
    inferred_language = ""
    if len(langs) > 0:
        counts = Counter(langs)
        top_lang, top_count = counts.most_common(1)[0]
        if top_count / len(langs) > LANGUAGE_THRESHOLD:
            inferred_language = "EN" if top_lang == "en-US" else "FR" if top_lang == "fr-FR" else ""
    return pd.Series({"Excluded_Course_IDs": excluded_ids, "Inferred_Language": inferred_language})

emp_signals = history.groupby("email_n").apply(derive_employee_signals)
employees = employees.merge(emp_signals, left_on="Email", right_index=True, how="left")
employees["Excluded_Course_IDs"] = employees["Excluded_Course_IDs"].apply(lambda x: x if isinstance(x, set) else set())
employees["Inferred_Language"] = employees["Inferred_Language"].fillna("")


## 5. Employee/course text + preference vocabulary gate

In [ ]:
employees["_text"] = (
    employees["Field_EN"].astype(str) + ". "
    + employees["Skills"].astype(str) + ". "
    + employees["Preferred_Formation"].astype(str)
).str.strip()

# PREFERENCE_VOCAB_BLOCKLIST, build_preference_vocab, and preference_is_informative
# now live in recommender_core.py (config-driven via recommender_config.yaml's
# preference_gate.vocab_blocklist), with their own pytest coverage in
# test_recommender_core.py -- no longer duplicated here.
PREFERENCE_VOCAB = core.build_preference_vocab(catalog_pool, config)
has_pref = employees["Preferred_Formation"].apply(
    lambda text: core.preference_is_informative(text, PREFERENCE_VOCAB)
).values
n_emp = len(employees)
print(f"Employees with an informative preference: {has_pref.sum()} / {n_emp}")

# Sparse-profile flag, for the section 10 verification -- not used in scoring itself,
# just to check afterward whether this group's outcomes actually changed.
employees["_skills_blank"] = employees["Skills"].astype(str).str.strip().eq("")
employees["_sparse_profile"] = employees["_skills_blank"] & ~pd.Series(has_pref, index=employees.index)
print(f"Sparse-profile employees (blank Skills AND no informative preference): "
      f"{employees['_sparse_profile'].sum()} / {n_emp} ({employees['_sparse_profile'].mean():.1%})")


## 6. Field match - TF-IDF-weighted token overlap (unchanged mechanism)

In [ ]:
# field_tokens / compute_field_score_norm now live in recommender_core.py
# (config-driven via recommender_config.yaml's field_match.min_field_token_len
# and field_match.field_stopwords), with their own pytest coverage -- no
# longer duplicated here.
field_score_norm = core.compute_field_score_norm(employees, catalog_pool, "_text", config)
print(f"field boost fires for {(field_score_norm > 0).mean():.1%} of (employee, course) pairs")


## 7. Boost weights and the rebalanced ranking function

Two changes from V3/the comparison notebook, both described in the intro:
`FIELD_BOOST` raised 0.03 -> 0.15, and the old flat domain-gate penalty
replaced by `GENERIC_BOOST_FLOOR` - a continuous multiplier applied to the
*positive* generic signals (level-match, language-match/default,
training-type) based on how much real topical signal (`field_score_norm`
+ preference match) exists for that specific (employee, course) pair.

- `topical_signal = clip(field_score_norm + (pref_norm if has_pref else 0), 0, 1)`
- `generic_scale = GENERIC_BOOST_FLOOR + (1 - GENERIC_BOOST_FLOOR) * topical_signal`
  -- ranges from `GENERIC_BOOST_FLOOR` (0.3) when there's zero topical
  signal for this pick, up to 1.0 (no reduction at all) when topical signal
  is strong. No threshold to dodge -- a small, spurious field-match score
  now produces a small, proportional reduction in the scale-down instead of
  either fully triggering or fully avoiding a flat penalty.
- **`LEVEL_MISMATCH_PENALTY` is deliberately NOT scaled** -- a level
  mismatch is real negative information regardless of topical signal
  strength, unlike the positive generic boosts, which are closer to a
  "free lunch" the rebalance is specifically targeting.
*(The weight values themselves -- `field_boost: 0.15`, `generic_boost_floor: 0.3`, `level_mismatch_penalty: 0.10`, etc. -- now live in `recommender_config.yaml`, not as notebook constants; this section just explains the mechanism.)*


In [ ]:
# All boost weights and the KIND_SENTENCES/STRONG_KINDS/sentence_for helpers
# now live in recommender_config.yaml and recommender_core.py respectively --
# see those files for the reasoning behind each weight. Printed here purely
# as a sanity check that the config loaded as expected.
print("Boost weights loaded from recommender_config.yaml:")
for k, v in config["boosts"].items():
    print(f"  {k}: {v}")


## 8. Embed with `Qwen3-Embedding-0.6B`

Per the model's own card: `prompt_name="query"` on the employee/preference
side (the "query"), no prompt on the course side (the "document"). Needs
`transformers>=4.51.0` / `sentence-transformers>=2.7.0`.

In [ ]:
def embed_qwen3(emp_texts, course_texts, pref_texts, has_pref):
    import torch
    from sentence_transformers import SentenceTransformer, util

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    model = SentenceTransformer(MODEL_NAME, device=device)

    emp_vectors = model.encode(list(emp_texts), batch_size=64, normalize_embeddings=True,
                                show_progress_bar=True, prompt_name="query")
    course_vectors = model.encode(list(course_texts), batch_size=64, normalize_embeddings=True,
                                   show_progress_bar=True)

    pref_vectors = np.zeros((len(emp_texts), emp_vectors.shape[1]), dtype=np.float32)
    has_pref_arr = np.asarray(has_pref)
    if has_pref_arr.any():
        pref_subset = [t for t, h in zip(pref_texts, has_pref_arr) if h]
        encoded = model.encode(pref_subset, batch_size=64, normalize_embeddings=True,
                                show_progress_bar=True, prompt_name="query")
        pref_vectors[has_pref_arr] = encoded

    sim_matrix = util.cos_sim(emp_vectors, course_vectors).numpy()
    pref_sim_matrix = util.cos_sim(pref_vectors, course_vectors).numpy()
    return sim_matrix, pref_sim_matrix


emp_texts = employees["_text"].tolist()
course_texts = catalog_pool["_text"].tolist()
pref_texts = employees["Preferred_Formation"].tolist()

sim_matrix, pref_sim_matrix = embed_qwen3(emp_texts, course_texts, pref_texts, has_pref)


## 9. Score, rank, build explanations, and save top-3 recommendations

In [ ]:
# compute_rank_score_matrix, select_top_k, and describe_pick now all live in
# recommender_core.py, with their own pytest coverage (test_recommender_core.py)
# -- the notebook's job from here is just to call them and save the result.
rank_score_matrix_masked, extras = core.compute_rank_score_matrix(
    sim_matrix, pref_sim_matrix, field_score_norm, has_pref, employees, catalog_pool, config
)

top_idx = core.select_top_k(rank_score_matrix_masked, TOP_K)

results = []
for i, emp in employees.iterrows():
    if rank_score_matrix_masked[i].max() == -np.inf:
        results.append({"Employee_ID": emp["Employee_ID"], "Email": emp["Email"], "Rank": None,
                         "Course_ID": None, "Course_Title": None, "similarity": None, "rank_score": None,
                         "reason": "no eligible courses remaining", "explanation": "no eligible courses remaining",
                         "flagged_for_review": True})
        continue
    for rank, j in enumerate(top_idx[i], start=1):
        rec = core.describe_pick(
            i, j, sim_matrix=sim_matrix, pref_sim_matrix=pref_sim_matrix,
            rank_score_matrix_masked=rank_score_matrix_masked, has_pref=has_pref,
            extras=extras, employees=employees, catalog_pool=catalog_pool, config=config,
        )
        results.append({"Employee_ID": emp["Employee_ID"], "Email": emp["Email"], "Rank": rank,
                         "Course_ID": rec["Course_ID"], "Course_Title": rec["Course_Title"],
                         "similarity": round(rec["similarity"], 4), "rank_score": round(rec["rank_score"], 4),
                         "reason": rec["reason"], "explanation": rec["explanation"],
                         "flagged_for_review": rec["flagged_for_review"]})

results_df = pd.DataFrame(results)
out_path = OUTPUTS / "recommendations_v4.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved {out_path}")

top1 = results_df[results_df["Rank"] == 1]
print(f"flagged_for_review rate: {top1['flagged_for_review'].mean():.1%}")
print(f"mean top-1 similarity: {top1['similarity'].mean():.3f}")


## 10. Verification - did the fix actually change what it targeted?

Reruns the three sanity-check cases and the sparse-profile blind-spot
measurement from the LLM-judge review, against **this run's** output, not
the old one. The question isn't "does this look nice" - it's specifically
"did the diagnosed mechanism actually change".

In [ ]:
KNOWN_HUB_TITLES = ["Careers in Commercial Banking",
                     "Strategic Vision and Decision-Making for Senior Executives and C-Suite Leaders",
                     "Creating Your IT Strategy"]

merged = top1.merge(employees[["Email", "Bu_new", "Employee_Level", "_sparse_profile"]], on="Email", how="left")

print("Top-1 hub concentration, this run:")
for title in KNOWN_HUB_TITLES:
    count = int((merged["Course_Title"] == title).sum())
    pct = count / len(merged) * 100
    print(f"  {title}: {count} ({pct:.1f}%)")

it_strategy = merged[merged["Course_Title"] == "Creating Your IT Strategy"]
if len(it_strategy) > 0:
    off_dept = it_strategy[~it_strategy["Bu_new"].str.contains("Technology", case=False, na=False)]
    print(f"\n'Creating Your IT Strategy' recipients outside Technology & Digital: "
          f"{len(off_dept)} / {len(it_strategy)}")

sparse_top1 = merged[merged["_sparse_profile"]]
print(f"\nSparse-profile employees (n={len(sparse_top1)}): top-1 course diversity")
print(f"  distinct courses recommended: {sparse_top1['Course_Title'].nunique()}")
print(f"  most concentrated pick: {sparse_top1['Course_Title'].value_counts().head(5).to_dict()}")


## 11. Before/after - direct comparison against V4's predecessor's output

Loads the actual `recommendations_translated_en_qwen3-embedding-0.6b.csv`
already saved on disk from `Recommend_Model_Comparison.ipynb` (same model,
same language condition, old scoring formula) and compares it directly
against this run's output - the cleanest way to see exactly what the
rebalance changed, not just whether the new numbers look reasonable in
isolation.

In [ ]:
if PREVIOUS_RECOMMENDATIONS_PATH.exists():
    previous_df = pd.read_csv(PREVIOUS_RECOMMENDATIONS_PATH)
    previous_top1 = previous_df[previous_df["Rank"] == 1]

    comparison_rows = []
    for title in KNOWN_HUB_TITLES:
        before = int((previous_top1["Course_Title"] == title).sum())
        after = int((top1["Course_Title"] == title).sum())
        comparison_rows.append({
            "course": title, "before_count": before, "before_pct": round(before / len(previous_top1) * 100, 1),
            "after_count": after, "after_pct": round(after / len(top1) * 100, 1),
        })

    comparison_df = pd.DataFrame(comparison_rows)
    comparison_path = OUTPUTS / "before_after_hub_comparison.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print(f"Saved {comparison_path}")
    print(comparison_df.to_string(index=False))

    print(f"\nCoverage before: {previous_top1['Course_Title'].nunique()} distinct top-1 courses / {n_course} pool")
    print(f"Coverage after:  {top1['Course_Title'].nunique()} distinct top-1 courses / {n_course} pool")
    print(f"\nflagged_for_review before: {previous_top1['flagged_for_review'].mean():.1%}")
    print(f"flagged_for_review after:  {top1['flagged_for_review'].mean():.1%}")
else:
    print(f"{PREVIOUS_RECOMMENDATIONS_PATH} not found -- skipping before/after comparison. "
          f"Run Recommend_Model_Comparison.ipynb's translated_en/qwen3 cell first, or point "
          f"COMPARISON_OUTPUTS at wherever that CSV actually lives.")
